# Gradient Accumulation

## Learning Objectives
1. Understand why gradient accumulation enables large effective batch sizes on memory-constrained hardware.
2. Simulate the accumulation algorithm in NumPy and verify the loss curve matches true large-batch training.
3. Implement gradient accumulation in a PyTorch training loop with correct zero_grad placement.
4. Combine gradient accumulation with mixed precision (AMP) and gradient clipping for production training.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Gradient Accumulation Simulation (NumPy)

In [ ]:
# Simulate accumulating gradients across N mini-batches before updating.
# Key insight: accumulated gradient == gradient on the full merged batch
# (when loss is mean-reduced and batch sizes are equal).

def mse_grad(X_batch, y_batch, w):
    """MSE gradient: dL/dw = (2/n) * X^T (Xw - y)."""
    residuals = X_batch @ w - y_batch
    return (2.0 / len(y_batch)) * X_batch.T @ residuals


np.random.seed(42)
n, d = 256, 5
X_data = np.random.randn(n, d)
true_w = np.array([1.0, -2.0, 0.5, 1.5, -1.0])
y_data = X_data @ true_w + 0.1 * np.random.randn(n)

accumulation_steps = 8   # simulate large batch = 32*8 = 256 samples
micro_batch_size = 32
lr = 0.05

w_accum = np.zeros(d)
w_large = np.zeros(d)   # baseline: full-batch gradient update

n_updates = 20
losses_accum = []
losses_large = []

for step in range(n_updates):
    # --- Gradient Accumulation (micro-batches) ---
    accum_grad = np.zeros(d)
    for acc_step in range(accumulation_steps):
        idx = np.random.randint(0, n, micro_batch_size)
        accum_grad += mse_grad(X_data[idx], y_data[idx], w_accum)
    accum_grad /= accumulation_steps  # normalise: average gradient across accumulated steps
    w_accum -= lr * accum_grad
    losses_accum.append(((X_data @ w_accum - y_data)**2).mean())

    # --- Large-Batch Baseline ---
    idx_large = np.random.randint(0, n, micro_batch_size * accumulation_steps)
    large_grad = mse_grad(X_data[idx_large], y_data[idx_large], w_large)
    w_large -= lr * large_grad
    losses_large.append(((X_data @ w_large - y_data)**2).mean())

print("Final MSE — Gradient Accumulation:", f"{losses_accum[-1]:.6f}")
print("Final MSE — Large Batch Baseline: ", f"{losses_large[-1]:.6f}")
print("The curves should be similar, confirming mathematical equivalence.")


## Level 2: PyTorch Training Loop with gradient_accumulation_steps

In [ ]:
# Correct gradient accumulation in PyTorch:
# 1. Do NOT call optimizer.zero_grad() inside the accumulation loop.
# 2. Average loss by dividing by accumulation_steps BEFORE .backward().
# 3. Call optimizer.step() + optimizer.zero_grad() only after all micro-batches.

ACCUMULATION_STEPS = 4
MICRO_BATCH = 32
EFFECTIVE_BATCH = MICRO_BATCH * ACCUMULATION_STEPS

torch.manual_seed(42)
X_t = torch.randn(800, 20, device=device)
w_true = torch.randn(20, 1, device=device)
y_t = X_t @ w_true + 0.05 * torch.randn(800, 1, device=device)

train_ds = TensorDataset(X_t[:640], y_t[:640])
val_ds = TensorDataset(X_t[640:], y_t[640:])
# Micro-batch loader
micro_loader = DataLoader(train_ds, batch_size=MICRO_BATCH, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64)


def build_net():
    return nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1)).to(device)


model = build_net()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

history_accum = []
for epoch in range(50):
    model.train()
    optimizer.zero_grad()   # clear at epoch start
    accumulated = 0
    running_loss = 0.0
    for step, (xb, yb) in enumerate(micro_loader):
        try:
            loss = criterion(model(xb), yb) / ACCUMULATION_STEPS  # scale loss
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                torch.cuda.empty_cache()
                print("OOM — reduce MICRO_BATCH")
                continue
            raise
        loss.backward()
        running_loss += loss.item()
        accumulated += 1
        if accumulated == ACCUMULATION_STEPS:
            optimizer.step()
            optimizer.zero_grad()
            accumulated = 0

    # Handle any remaining incomplete accumulation group
    if accumulated > 0:
        optimizer.step()
        optimizer.zero_grad()

    model.eval()
    with torch.no_grad():
        vl = sum(criterion(model(xb), yb).item() * len(xb) for xb, yb in val_loader) / len(val_loader.dataset)
    history_accum.append(vl)

print(f"Final val MSE (accum, effective_batch={EFFECTIVE_BATCH}): {history_accum[-1]:.6f}")


## Real-World Example 1: Effective Batch Size Comparison

In [ ]:
# Compare: small batch, large batch, and small batch + accumulation
# Expected: accumulation curve ≈ large batch curve

torch.manual_seed(42)
criterion_rw = nn.MSELoss()


def train_fixed(batch_size, n_epochs=60):
    """Train with fixed batch size, no accumulation."""
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    m = build_net()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    hist = []
    for _ in range(n_epochs):
        m.train()
        for xb, yb in loader:
            opt.zero_grad()
            criterion_rw(m(xb), yb).backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            vl = sum(criterion_rw(m(xb), yb).item() * len(xb) for xb, yb in val_loader) / len(val_loader.dataset)
        hist.append(vl)
    return hist


def train_with_accumulation(micro_bs, acc_steps, n_epochs=60):
    """Train with gradient accumulation (micro_bs * acc_steps = effective batch)."""
    loader = DataLoader(train_ds, batch_size=micro_bs, shuffle=True, drop_last=True)
    m = build_net()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    hist = []
    for _ in range(n_epochs):
        m.train()
        opt.zero_grad()
        acc = 0
        for xb, yb in loader:
            (criterion_rw(m(xb), yb) / acc_steps).backward()
            acc += 1
            if acc == acc_steps:
                opt.step(); opt.zero_grad(); acc = 0
        if acc > 0:
            opt.step(); opt.zero_grad()
        m.eval()
        with torch.no_grad():
            vl = sum(criterion_rw(m(xb), yb).item() * len(xb) for xb, yb in val_loader) / len(val_loader.dataset)
        hist.append(vl)
    return hist


h_small = train_fixed(batch_size=8)
h_large = train_fixed(batch_size=64)
h_accum = train_with_accumulation(micro_bs=8, acc_steps=8)   # same effective batch as h_large

print(f"Small batch (8)      final val MSE: {h_small[-1]:.6f}")
print(f"Large batch (64)     final val MSE: {h_large[-1]:.6f}")
print(f"Accumulation (8×8)   final val MSE: {h_accum[-1]:.6f}")
print("Accumulation ≈ Large batch (demonstrates mathematical equivalence)")


## Real-World Example 2: Mixed Precision + Gradient Accumulation

In [ ]:
# AMP autocast reduces memory and speeds up compute on CUDA GPUs.
# GradScaler prevents gradient underflow with fp16.
# Accumulation must scale loss and call scaler.unscale_ before clipping.

torch.manual_seed(42)
model_amp = build_net()
optimizer_amp = torch.optim.Adam(model_amp.parameters(), lr=1e-3)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
AMP_ACC_STEPS = 4
criterion_amp = nn.MSELoss()
loader_amp = DataLoader(train_ds, batch_size=16, shuffle=True, drop_last=True)

history_amp = []
for epoch in range(40):
    model_amp.train()
    optimizer_amp.zero_grad()
    acc = 0
    for xb, yb in loader_amp:
        # autocast: ops run in fp16 on CUDA, fp32 on CPU
        with torch.autocast(device_type=device.type, enabled=torch.cuda.is_available()):
            try:
                loss = criterion_amp(model_amp(xb), yb) / AMP_ACC_STEPS
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    torch.cuda.empty_cache()
                    continue
                raise
        scaler.scale(loss).backward()
        acc += 1
        if acc == AMP_ACC_STEPS:
            scaler.unscale_(optimizer_amp)          # unscale before clipping
            torch.nn.utils.clip_grad_norm_(model_amp.parameters(), max_norm=1.0)
            scaler.step(optimizer_amp)
            scaler.update()
            optimizer_amp.zero_grad()
            acc = 0
    if acc > 0:
        scaler.step(optimizer_amp); scaler.update(); optimizer_amp.zero_grad()

    model_amp.eval()
    with torch.no_grad():
        vl = sum(criterion_amp(model_amp(xb), yb).item() * len(xb) for xb, yb in val_loader) / len(val_loader.dataset)
    history_amp.append(vl)

print(f"AMP + accumulation final val MSE: {history_amp[-1]:.6f}")
print(f"AMP scaler state: scale={scaler.get_scale():.1f}  (should remain >=1)")


## Real-World Example 3: Gradient Clipping with Accumulation

In [ ]:
# Gradient clipping prevents exploding gradients.
# With accumulation, you MUST clip AFTER accumulating (before optimizer.step).
# Clipping before accumulation clips each micro-batch separately — wrong.

torch.manual_seed(42)
# Use a deeper network that is more prone to gradient explosion
model_clip = nn.Sequential(
    nn.Linear(20, 128), nn.Tanh(),
    nn.Linear(128, 128), nn.Tanh(),
    nn.Linear(128, 64), nn.Tanh(),
    nn.Linear(64, 1),
).to(device)
optimizer_clip = torch.optim.SGD(model_clip.parameters(), lr=0.05, momentum=0.9)
criterion_clip = nn.MSELoss()
CLIP_NORM = 1.0
ACC_STEPS = 4
loader_clip = DataLoader(train_ds, batch_size=16, shuffle=True, drop_last=True)

grad_norms = []  # track gradient norms before clipping
val_losses = []

for epoch in range(50):
    model_clip.train()
    optimizer_clip.zero_grad()
    acc = 0
    for xb, yb in loader_clip:
        (criterion_clip(model_clip(xb), yb) / ACC_STEPS).backward()
        acc += 1
        if acc == ACC_STEPS:
            # Compute gradient norm BEFORE clipping for monitoring
            total_norm = torch.nn.utils.clip_grad_norm_(
                model_clip.parameters(), max_norm=CLIP_NORM
            )
            grad_norms.append(total_norm.item())
            optimizer_clip.step()
            optimizer_clip.zero_grad()
            acc = 0
    if acc > 0:
        torch.nn.utils.clip_grad_norm_(model_clip.parameters(), CLIP_NORM)
        optimizer_clip.step(); optimizer_clip.zero_grad()

    model_clip.eval()
    with torch.no_grad():
        vl = sum(criterion_clip(model_clip(xb), yb).item() * len(xb) for xb, yb in val_loader) / len(val_loader.dataset)
    val_losses.append(vl)

clipped_count = sum(1 for g in grad_norms if g > CLIP_NORM)
print(f"Total gradient norm measurements: {len(grad_norms)}")
print(f"Steps where gradient was clipped: {clipped_count} ({100*clipped_count/len(grad_norms):.1f}%)")
print(f"Max pre-clip gradient norm: {max(grad_norms):.4f}")
print(f"Mean pre-clip gradient norm: {np.mean(grad_norms):.4f}")
print(f"Final val MSE: {val_losses[-1]:.6f}")
print("Rule: clip AFTER accumulation, BEFORE optimizer.step()")
